# Сравнение функций потерь CatBoost

Подключите исходные parquet, результаты этапа 00 и выход этапа 03. Включите GPU.

Сравниваем четыре функции потерь с выбранными на этапе 03 окном и признаками. Для MAE используем готовый результат. Первые две недели оценивают аддитивную поправку, следующие две служат для выбора, последние две для проверки победителя.

Для работы частями задайте `MAX_NEW_FITS` и сохраните папку этапа. В новой сессии укажите её в `RESTORE_DIR`. Финальные модели обучаются на всех доступных датах выбранного окна.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path.cwd()

def locate(name):
    paths = sorted(INPUT_ROOT.rglob(name))
    if len(paths) != 1:
        raise ValueError(f"Ожидался один {name}. Найдено: {paths}")
    return paths[0]

baseline = locate("weekly_predictions.parquet").parent
control = pd.read_parquet(baseline / "weekly_predictions.parquet")
future_control = pd.read_parquet(baseline / "full_predictions.parquet")
run_id = control.run_id.unique().item()
keys = ["cutoff","horizon","route_id"]
def validate(panel, template, labeled):
    cols = keys + (["y","period"] if labeled else ["id"])
    a = template[cols].merge(panel[cols], how="outer", on=keys, validate="one_to_one", indicator=True)
    if not a._merge.eq("both").all() or any(not a[k+"_x"].eq(a[k+"_y"]).all() for k in cols[len(keys):]):
        raise ValueError("Не совпадают ключи, метки или периоды прогнозов")
    if not np.isfinite(panel.raw_prediction).all():
        raise ValueError("Неконечный raw_prediction")

validate(control,control,True)
validate(future_control,future_control,False)
if not control.groupby("cutoff").size().eq(8000).all() or control.cutoff.nunique()!=42 or len(future_control)!=8000:
    raise ValueError("Ожидалась полная панель baseline")
if not control.model_info_through.lt(control.cutoff).all():
    raise ValueError("Некорректные даты обучения контроля")

In [ ]:
EXPERIMENT = "04_losses"
CODE_VERSION = 'a90ba6ab53be65fb'
import time
import gc
import hashlib
import shutil
import pyarrow as pa
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count

GPU_DEVICES = "0"
MAX_NEW_FITS = None
RESTORE_DIR = None
ITERATIONS = 1500
TARGET_SCALE = 100000
STEP = pd.Timedelta(minutes=30)
TARGET = "target_1h"
STATUSES = [f"status_{i}" for i in range(1,7)]
TRAIN_FILE = "train_solo_track.parquet"
TEST_FILE = "test_solo_track.parquet"
TRAIN_PATH, TEST_PATH = locate(TRAIN_FILE), locate(TEST_FILE)
if get_gpu_device_count() < 1:
    raise RuntimeError("Включите GPU в Kaggle")
manifest = json.loads((baseline/"manifest.json").read_text())
for path in [TRAIN_PATH, TEST_PATH]:
    with path.open("rb") as stream:
        if hashlib.sha256(stream.read()).hexdigest()!=manifest["input_hashes"][path.name]:
            raise ValueError(f"Исходные данные отличаются от baseline: {path.name}")
ROOT = Path.cwd() / EXPERIMENT
if RESTORE_DIR is not None and not ROOT.exists():
    shutil.copytree(Path(RESTORE_DIR), ROOT)
ROOT.mkdir(exist_ok=True)
cache = ROOT / "cache"
cache.mkdir(exist_ok=True)
contract = dict(code=CODE_VERSION, baseline=run_id, iterations=ITERATIONS,
                gpu=GPU_DEVICES, catboost=__import__("catboost").__version__)
marker = ROOT / "contract.json"
if marker.exists() and json.loads(marker.read_text()) != contract:
    raise ValueError("Checkpoint относится к другим настройкам")
marker.write_text(json.dumps(contract, indent=2))
new_fits = 0
class FitBudgetReached(Exception):
    pass

def partial_stop():
    print(f"Часть завершена. Сохраните Output {ROOT}, включая cache. Продолжение через RESTORE_DIR.")

In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=["route_id", "timestamp", TARGET, *STATUSES]).sort_values(["timestamp", "route_id"])
test = pd.read_parquet(TEST_PATH, columns=["id", "route_id", "timestamp"])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError("В данных есть пропуски")
if not (not history.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторяется ключ маршрута и времени")
if not (times.equals(pd.date_range(times[0], times[-1], freq=STEP))):
    raise ValueError("Нарушена получасовая сетка")
if not (len(history) == len(times) * len(routes)):
    raise ValueError("Неполная панель маршрутов")
if not (test.id.is_unique and not test.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторы идентификаторов или ключей теста")
end = times[-1]
if not ((end.hour, end.minute) == (10, 30)):
    raise ValueError("Последняя точка истории должна быть в 10:30")
expected = pd.MultiIndex.from_product([routes, [end + h*STEP for h in range(1, 9)]],
                                      names=["route_id", "timestamp"])
if not (expected.difference(pd.MultiIndex.from_frame(test[["route_id", "timestamp"]])).empty):
    raise ValueError("Тест не покрывает все ожидаемые маршруты и горизонты")
if not (len(test) == len(expected)):
    raise ValueError("Число строк теста отличается от ожидаемого")
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes))
          for name in [TARGET, *STATUSES]}
del history
if not all(np.isfinite(array).all() for array in values.values()):
    raise ValueError("В числовых рядах есть бесконечные значения")

dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 42:
    raise ValueError("Недостаточно дат для временного разбиения")
tune_start, holdout_start = dates[-28], dates[-14]
LAG_VALIDATION_FIT_BEFORE = tune_start.normalize()

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

means = {(name, steps): rolling(array, steps)
         for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + "_hourly"] = values[name] + past(values[name], positions - 1)
extra_signals["warehouse_total_load"] = sum(extra_signals[name + "_hourly"] for name in STATUSES[:3])
extra_signals["warehouse_processing_rate"] = (
    extra_signals["status_3_hourly"] / (extra_signals["status_1_hourly"] + np.float32(1e-5)))
for name in STATUSES[3:]:
    extra_signals[name + "_diff_hourly"] = values[name] - past(values[name], positions - 2)
extra_signals["hub_buffer"] = values["status_5"] - values["status_4"]
extra_signals["hub_gate_bottleneck"] = (
    extra_signals["status_4_diff_hourly"] - extra_signals["status_6_diff_hourly"])
extra_signals["target_acceleration"] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals["status_6_diff_hourly"]

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f"target_back_{k}"] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f"{name}_mean_{steps}"] = means[name, steps][origins]
        if name in STATUSES:
            common[f"{name}_latest"] = array[origins]
            if name in STATUSES[:3]:
                common[f"{name}_change_1h"] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f"target_1h_lag_{lag}"] = past(target, origins - lag)
    common["warehouse_total_load_lag_48"] = past(extra_signals["warehouse_total_load"], origins - 48)
    common["status_6_diff_hourly_lag_48"] = past(hub_signal, origins - 48)
    common["target_vs_mean_6h"] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day*48) for day in range(1, 8)])
    common["morning_level_ratio"] = np.divide(
        recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common

CORRELATION_COLUMNS = ["selected_status6_lag_steps", "selected_status6_correlation",
                       "selected_status6_pair_count", "status6_lag_valid",
                       "lagged_status6_signal", "lagged_status6_signal_available"]
FEATURE_GROUPS = {
    "calendar": ["hour", "minute", "hour_float", "hour_sin", "hour_cos", "dayofweek",
                 "is_weekend", "weekday_sin", "weekday_cos"],
    "local_statuses": ["status_1_hourly", "status_2_hourly", "status_3_hourly",
                       "warehouse_total_load", "warehouse_processing_rate"],
    "hub_combinations": ["status_4_diff_hourly", "status_5_diff_hourly", "status_6_diff_hourly",
                         "hub_buffer", "hub_gate_bottleneck"],
    "historical_lags": ["target_1h_lag_48", "target_1h_lag_96",
                        "warehouse_total_load_lag_48", "status_6_diff_hourly_lag_48"],
    "target_change": ["target_acceleration"],
    "correlation_lag": CORRELATION_COLUMNS,
    "pilot_start_time": ["start_sin", "start_cos"],
    "pilot_peaks": ["peak7_mean_max", "peak7_mean_level",
                    "peak7_share_-1", "peak7_share_0", "peak7_share_1"],
}

In [ ]:
def add_correlation_features(frame, h, lag_table, chunk_rows=200_000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError("Некорректный горизонт или размер блока")
    by_route = lag_table.set_index("route_id").reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError("Строка признаков не сопоставлена с историей")
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = (np.isfinite(selected) & (selected >= h) & (source >= 0)
                     & (source <= origin) & (source < len(times)))
        signal = hub_signal[np.clip(source, 0, len(times)-1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions["lagged_status6_signal"]] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions["lagged_status6_signal_available"]] = available.astype(np.float32)
    return frame

In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({
        "cutoff": times[origins].repeat(len(routes)),
        "timestamp": dates.repeat(len(routes)),
        "route_id": np.tile(routes, len(origins)),
    })
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {
        "hour": dates.hour, "minute": dates.minute,
        "hour_float": dates.hour + dates.minute / 60.0,
        "dayofweek": dates.dayofweek, "is_weekend": (dates.dayofweek >= 5).astype(np.int8),
    }
    calendar["hour_sin"] = np.sin(2 * np.pi * calendar["hour_float"] / 24)
    calendar["hour_cos"] = np.cos(2 * np.pi * calendar["hour_float"] / 24)
    calendar["weekday_sin"] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar["weekday_cos"] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    for name in CORRELATION_COLUMNS:
        frame[name] = np.float32(np.nan)
    for days in (1, 2, 7):
        frame[f"slot_days_back_{days}"] = past(target, slot - days*48).ravel()
    for name, days in [("slot_daily7", range(1, 8)), ("slot_weekly4", (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d*48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day*48)
        adjusted = past(target, slot - day*48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame["profile28"] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = np.maximum(0, total - upper), np.minimum(total, total - lower)
        seasonal = frame[["slot_daily7", "slot_weekly4"]].mean(axis=1).to_numpy()
        frame["overlap"] = np.where((lower <= upper).ravel(),
                                    (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame["overlap"] = np.nan
    numeric = frame.columns.difference(["cutoff", "timestamp", "route_id"])
    return frame.astype({name: "float32" for name in numeric})

In [ ]:
def metrics(frame, prediction):
    y = frame.y.to_numpy(dtype=np.float64)
    p = np.asarray(prediction, dtype=np.float64)
    if p.shape != y.shape or not np.isfinite(p).all() or not np.isfinite(y).all():
        raise ValueError('Некорректные факты или прогнозы')
    e = p - y
    daily = pd.DataFrame(dict(day=frame.cutoff.dt.normalize().to_numpy(), y=y,
                              absolute=np.abs(e), signed=e)).groupby('day').sum()
    if (daily.y <= 0).any():
        raise ValueError('Неопределённая метрика: нулевой объём дня')
    return dict(rows=len(frame), days=len(daily), WAPE=float(daily.absolute.sum()/daily.y.sum()),
                signed_bias=float(daily.signed.sum()/daily.y.sum()),
                score=float((daily.absolute.sum()+abs(daily.signed.sum()))/daily.y.sum()),
                daily_score=float(((daily.absolute+daily.signed.abs())/daily.y).mean()))

def atomic_parquet(frame, path):
    temporary = path.with_suffix('.partial.parquet')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)


prior_root = locate("03_windows_features_selection.json").parent
prior_spec = tuple(json.loads((prior_root/"03_windows_features_selection.json").read_text())["spec"])
prior_history = pd.read_parquet(prior_root/"03_windows_features_history.parquet")
prior_future = pd.read_parquet(prior_root/"03_windows_features_future.parquet")
validate(prior_history,control,True)
validate(prior_future,future_control,False)
weekly_days = pd.DatetimeIndex(control.cutoff.unique()).sort_values()
stages = {f"week{i+1}":weekly_days[i*7:(i+1)*7] for i in range(6)}
stages["full"] = pd.DatetimeIndex([end])
base_spec = ("recent75","base","MAE")
key_cols = ["cutoff","horizon","route_id","timestamp","raw_prediction","model_info_through"]
network = {name:values[name].sum(axis=1,dtype=np.float64) for name in [TARGET,*STATUSES]}

def variant_features(frame, h, group):
    if h < 3 or group == "base":
        return frame
    origins = times.get_indexer(pd.DatetimeIndex(frame.cutoff.unique()))
    if group in ["peaks","peaks_network"]:
        peaks, ranges, slopes = [], [], []
        for day in range(1,8):
            center = origins + h - 48*day
            left, mid, right = (past(target, center+j) for j in (-1,0,1))
            hi = np.maximum(np.maximum(left,mid),right)
            peaks.append(hi)
            ranges.append(hi-np.minimum(np.minimum(left,mid),right))
            slopes.append(right-left)
        for name, arrays in [("max",peaks),("range",ranges),("slope",slopes)]:
            frame["peak7_"+name] = average_available(arrays).ravel()
    if group in ["network","peaks_network"]:
        for name, arr in network.items():
            frame["network_"+name] = np.repeat(arr[origins],len(routes)).astype("float32")
            frame["network_change_"+name] = np.repeat(arr[origins]-past(arr,origins-2),len(routes)).astype("float32")
    return frame

def history_matrix(h, group):
    temp = ROOT / f"features_h{h}.partial.parquet"
    writer = None
    try:
        for start in range(0,len(times)-h,48):
            ix = np.arange(start,min(start+48,len(times)-h))
            frame = variant_features(horizon_dataset(h,ix),h,group)
            frame["y"] = target[ix+h].ravel()
            table = pa.Table.from_pandas(frame,preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temp,table.schema,compression="zstd")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    frame = pd.read_parquet(temp)
    temp.unlink()
    return frame

def effective(spec,h):
    window,group,loss = spec
    return window,group if h>=3 else "base",loss

def get_task(data,spec,h,stage):
    global new_fits
    window,group,loss = spec
    tag = "_".join([window,group,loss.replace(":","_").replace("=","_"),stage,f"h{h}"])
    path = cache / (tag+".parquet")
    log = cache / (tag+".json")
    if spec==effective(prior_spec,h):
        panel = prior_future if stage=="full" else prior_history.loc[prior_history.cutoff.isin(stages[stage])]
        return panel.loc[panel.horizon.eq(h)].copy()
    if spec==base_spec:
        panel = future_control if stage=="full" else control.loc[control.stage.eq(stage)]
        return panel.loc[panel.horizon.eq(h)].copy()
    if log.exists() and path.is_file():
        return pd.read_parquet(path)
    if MAX_NEW_FITS is not None and new_fits >= MAX_NEW_FITS:
        raise FitBudgetReached()
    dates = stages[stage]
    boundary = end+STEP if stage=="full" else dates[0].normalize()
    available = data.timestamp < boundary
    if window=="recent75":
        cutoffs = pd.DatetimeIndex(data.loc[available,"cutoff"].unique()).sort_values()
        first = cutoffs[len(cutoffs)//4]
    else:
        first = boundary-pd.Timedelta(days=int(window))
    mask = available & data.cutoff.ge(first)
    lags = pd.read_csv(baseline/"lags"/(stage+".csv"))
    train = add_correlation_features(data.loc[mask].copy(),h,lags)
    if stage=="full":
        query = variant_features(horizon_dataset(h,np.array([len(times)-1])),h,group)
    else:
        query = data.loc[data.cutoff.isin(dates)].copy()
    query = add_correlation_features(query,h,lags)
    cols = [c for c in train if c not in ["cutoff","timestamp","profile28","overlap","y"]]
    model = CatBoostRegressor(iterations=ITERATIONS,depth=7,learning_rate=0.05,loss_function=loss,
                              cat_features=["route_id"],task_type="GPU",devices=GPU_DEVICES,
                              random_seed=42,thread_count=4,allow_writing_files=False,verbose=300)
    started = time.perf_counter()
    model.fit(train[cols],train.y/TARGET_SCALE)
    pred_train = np.maximum(model.predict(train[cols])*TARGET_SCALE,0)
    report = [dict(split="train",**metrics(train,pred_train))]
    prediction = np.maximum(model.predict(query[cols])*TARGET_SCALE,0)
    raw = prediction if h==1 else 0.7*prediction + 0.3*query.profile28
    part = query[["route_id","timestamp","cutoff"]].assign(horizon=h,raw_prediction=np.asarray(raw),
                                                          model_info_through=train.timestamp.max())
    if stage=="full":
        part = part.merge(test[["id","route_id","timestamp"]],on=["route_id","timestamp"],validate="one_to_one")
        model_dir = ROOT / "models"
        model_dir.mkdir(exist_ok=True)
        model_path = model_dir / (tag+".cbm")
        temporary = model_path.with_suffix(".partial.cbm")
        model.save_model(str(temporary))
        temporary.replace(model_path)
        for week in ["week5","week6"]:
            previous = json.loads((cache/(tag.replace("_full_","_"+week+"_")+".json")).read_text())
            report.append(dict(previous[1],split=week+" до refit"))
    else:
        part["y"] = query.y.to_numpy()
        part["period"] = "warmup" if dates[0]<tune_start else "tuning" if dates[0]<holdout_start else "holdout"
        report.append(dict(split="test окна "+stage,**metrics(query,prediction)))
    if len(part)!=len(dates)*len(routes) or part.duplicated(keys).any() or not np.isfinite(part.raw_prediction).all():
        raise ValueError("Некорректная панель")
    for row in report:
        row.setdefault("seconds", round(time.perf_counter()-started,2))
    display(pd.DataFrame(report).assign(horizon=h,stage=stage,window=window,group=group,loss=loss))
    atomic_parquet(part,path)
    tmp=log.with_suffix(".partial")
    tmp.write_text(json.dumps(report,ensure_ascii=False,indent=2))
    tmp.replace(log)
    new_fits += 1
    return part

def run_spec(spec, stage_names):
    panels = []
    for h in range(1,9):
        current = effective(spec,h)
        tag = "_".join([current[0],current[1],current[2].replace(":","_").replace("=","_")])
        missing = any(not (cache / f"{tag}_{s}_h{h}.json").exists() or not (cache / f"{tag}_{s}_h{h}.parquet").exists() for s in stage_names)
        data = history_matrix(h,current[1]) if current not in (base_spec,effective(prior_spec,h)) and missing else None
        for stage in stage_names:
            part = get_task(data,current,h,stage)
            keep = key_cols + (["id"] if stage=="full" else ["y","period"])
            panels.append(part[keep])
        del data
        gc.collect()
    out = pd.concat(panels,ignore_index=True)
    template = future_control if stage_names==["full"] else control.loc[control.stage.isin(stage_names)]
    validate(out,template,stage_names!=["full"])
    return out

def offset(frame, period):
    part = frame.loc[frame.period.isin(period)]
    return float((part.y-part.raw_prediction).mean())

def calibrated(frame,b):
    return np.maximum(0,frame.raw_prediction+b)

def assess(frame,b,period):
    rows = frame.loc[frame.period.eq(period)]
    return metrics(rows,calibrated(rows,b))

ranking = []
def choose(specs, step):
    candidates = []
    for spec in specs:
        part = run_spec(spec,["week1","week2","week3","week4"])
        b = offset(part,["warmup"])
        row = dict(step=step,window=spec[0],group=spec[1],loss=spec[2],offset=b,
                   raw_score=assess(part,0,"tuning")["daily_score"],**assess(part,b,"tuning"))
        for i,period in enumerate(["week3","week4"],1):
            week = part.loc[part.cutoff.isin(stages[period])]
            row[f"week{i}"] = metrics(week,calibrated(week,b))["daily_score"]
        ranking.append(row)
        candidates.append((row["daily_score"],spec))
    display(pd.DataFrame(ranking))
    pd.DataFrame(ranking).to_csv(ROOT/"tuning.csv",index=False)
    return min(candidates,key=lambda x:x[0])[1]

## Выбор по tuning


In [ ]:
winner = None
try:
    selected_path = locate("03_windows_features_selection.json")
    previous = json.loads(selected_path.read_text())
    if previous["run_id"] != run_id:
        raise ValueError("Этап 03 относится к другому запуску")
    window, group, _ = previous["spec"]
    winner = choose([(window,group,loss) for loss in ["MAE","Quantile:alpha=0.55","Quantile:alpha=0.60","RMSE"]],"loss")
except FitBudgetReached:
    partial_stop()

## Оценка до refit и финальный прогноз


In [ ]:
if winner is not None:
    try:
        panel = run_spec(winner,[f"week{i}" for i in range(1,7)])
        b = offset(panel,["warmup"])
        panel["y_pred"] = calibrated(panel,b)
        panel["run_id"] = run_id
        atomic_parquet(panel,ROOT/(EXPERIMENT+"_history.parquet"))
        records = []
        for name, data in [("weekly",control),("winner",panel)]:
            for period in ["tuning","holdout"]:
                part = data.loc[data.period.eq(period)]
                for group,rows in [("all",part),("week1",part.loc[part.cutoff.isin(sorted(part.cutoff.unique())[:7])]),
                                   ("week2",part.loc[part.cutoff.isin(sorted(part.cutoff.unique())[7:])]),
                                   ("Saturday",part.loc[part.cutoff.dt.dayofweek.eq(5)])]:
                    records.append(dict(model=name,period=period,group=group,**metrics(rows,rows.y_pred)))
        report = pd.DataFrame(records)
        display(report)
        report.to_csv(ROOT/"comparison.csv",index=False)
        print("Независимая оценка локального test получена до финального переобучения. Даты ранее изучались.")
        future = run_spec(winner,["full"])
        final_offset = offset(panel,["tuning","holdout"])
        if winner==base_spec:
            final_offset = float(json.loads((baseline/"reports"/"calibration.json").read_text())["submission_offset"])
        future["y_pred"] = calibrated(future,final_offset)
        future["run_id"] = run_id
        atomic_parquet(future,ROOT/(EXPERIMENT+"_future.parquet"))
        selection = dict(run_id=run_id, spec=list(winner), weekly_offset=b, submission_offset=final_offset)
        (ROOT/(EXPERIMENT+"_selection.json")).write_text(json.dumps(selection,indent=2))
    except FitBudgetReached:
        partial_stop()